# ALD — 전처리 · 헤드 학습 · 센서별 평가

`Python (PECVD)` 커널에서 위에서부터 실행합니다. 다른 노트북을 실행할 필요가 없습니다.
GEVIS 인코더·Fusion은 고정하고 ALD 판별 헤드만 학습합니다.

입력에는 연속형 센서와 밸브 상태가 함께 들어갑니다. **질문과 판별의 대상은 연속형 센서**이고,
`ValveAct_...` OPEN/CLOSE 상태는 **문맥 입력**입니다. 밸브 상태에는 이상을 주입하지 않습니다.
기존 세 질문(변화 여부·유형·정도)을 센서마다 반복하므로 질문을 추가할 필요가 없습니다.


In [ ]:
from pathlib import Path
import os, sys
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
WORKSPACE_PATH = None  # 데이터·가중치가 다른 폴더에 있으면 해당 경로 지정
SOURCE_ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents)
           if (p/'src/pecvd/gevis.py').is_file()),None)
if SOURCE_ROOT is None:
    raise FileNotFoundError('GEVIS 프로젝트 폴더에서 실행하세요.')
ROOT=SOURCE_ROOT.resolve() if WORKSPACE_PATH is None else Path(WORKSPACE_PATH).expanduser().resolve()
SOURCE=(SOURCE_ROOT/'src').resolve()
if not (ROOT/'config/experiments/gevis_pipeline.json').is_file() or not (ROOT/'artifacts').is_dir():
    raise FileNotFoundError('설정·가중치가 필요합니다. README를 확인하고 WORKSPACE_PATH에 준비한 작업 폴더를 지정하세요.')
for name,module in tuple(sys.modules.items()):
    if name=='pecvd' or name.startswith('pecvd.'):
        for location in [getattr(module,'__file__',None),*getattr(module,'__path__',())]:
            if location and not Path(location).resolve().is_relative_to(SOURCE):
                raise RuntimeError('다른 프로젝트의 pecvd가 로드되어 있습니다. 새 커널로 실행하세요.')
if str(SOURCE) not in sys.path:sys.path.insert(0,str(SOURCE))
import pandas as pd
from pecvd.gevis import load_gevis
from pecvd import gevis_training as training


## 실험 설정

`ENCODER='DEP.'`는 CVD DEP 인코더 하나를 ALD의 모든 지원 구간에 공유한다는 뜻입니다.
ALD의 DEP 구간만 선택하는 설정은 아닙니다. 이 실험에서는 인코더를 고정하고 세 헤드를 비교합니다.
DEP 인코더가 ALD에 최적인지는 이 설정만으로 알 수 없습니다.

파일 개수·월·레시피에 따른 사전 제외는 하지 않습니다. `data_complete` 자체에서만 Run을 분할하며 native 품질 검사를 적용합니다.

외부 전달용 입력·경계·정답·안내문은 전처리, 정규화, 학습, Validation 선택에 사용하지 않습니다.


In [ ]:
ENCODER='DEP.'
PREPROCESSING=training.PreprocessingConfig(
    schema_columns=None,       # 열 개수로 파일 제외 없음; 이름이 같은 공통 컬럼 사용
    max_window_length=64,
    encode_valves=True,        # OPEN/CLOSE 또는 기존 0/1 → 0/1 문맥 입력
    drop_columns=(),           # 추가로 입력에서 뺄 컬럼
    context_columns=(),        # ValveAct_ 외에 판별 대상에서 뺄 문맥 컬럼
)
NATIVE=training.NativeConfig(excluded_months=(),recipe_suffix='')
TRAINING=training.TrainingConfig(epochs=40,patience=8,batch_size=256,max_seconds=600.)
CHOICE_GROUPS=((1,3),(2,),(4,),(5,))
CHOICES=[training.ChoiceConfig(mode=mode,groups=CHOICE_GROUPS,sigmoid_threshold=0.5)
         for mode in ('softmax','grouped','sigmoid')]
# 1 Mean level / 2 Variability / 3 Trend slope / 4 Local change / 5 Delay
TARGET_PHASES=None             # None: 전체 지원 공정 경로
MAX_PREPARATION_MEMORY_MB=4096  # 한 파일 처리용 RAM 예산; 전체 원본은 디스크 캐시


In [ ]:
GEVIS=load_gevis(ROOT,encoder_steps=[ENCODER])
display(GEVIS.describe())
print('고정 인코더:',ENCODER,'| ALD 입력 구간:',TARGET_PHASES or '전체 지원 경로')


## ALD 전처리

1. `data/mt_nit/data_complete`의 CSV를 모두 입력 목록에 포함합니다. 파일별 열 개수가 달라도 제외하지 않습니다.
2. 파일 간 이름이 같은 공통 컬럼을 사용합니다. 다른 이름의 센서를 임의로 합치거나, 없는 값을 0으로 채우지 않습니다.
3. ID·시간·임의 텍스트는 센서 입력에서 제외합니다. 공정 메타데이터는 구간·Run 구분에만 사용합니다.
4. TRAIN에서 비수치·상수·관측 부족 센서를 경로별로 제외합니다. 연속형 센서는 TRAIN median/IQR로 정규화합니다.
5. `ValveAct_...`의 CLOSE/OPEN 및 기존 0/1은 0/1로 통일합니다. 상태 열은 학습 정답·이상 주입·판별 대상에서 제외하고 문맥 입력으로 유지합니다.
6. native Main/SubCycle·시간순 Run 분할·시간/센서 품질 검사를 적용합니다. 모든 파일을 읽어도 모든 구간이 학습에 쓰이는 것은 아닙니다.

밸브 개도·각도처럼 **연속값을 측정하는 센서**는 OPEN/CLOSE 상태 열과 구분합니다.
사용·제외 센서와 문맥/판별 역할은 다음 셀의 실제 결과로 확인합니다. 원본 CSV는 수정하지 않습니다.

원본 센서 값은 파일별로 `.cache/ald-native-inputs`에 저장하고 필요한 구간·열만 읽습니다.
173개 CSV의 DataFrame을 RAM에 누적하지 않습니다. 파일·컬럼 선택, 정규화와 분할 기준은 같습니다.
이전 MemoryError가 발생했던 커널은 재시작한 뒤 위에서부터 실행하세요.


In [ ]:
COMMON_COLUMNS,FILE_AUDIT=training.schema_inventory(ROOT/'data/mt_nit/data_complete',PREPROCESSING)
display(FILE_AUDIT.groupby(['columns','included','reason']).size().rename('files').reset_index())
print('입력 파일:',int(FILE_AUDIT.included.sum()),'/',len(FILE_AUDIT),'| 공통 컬럼:',len(COMMON_COLUMNS))
# 파일별 상세: FILE_AUDIT


In [ ]:
PREPARED=training.prepare_ald(ROOT,GEVIS,encoder=ENCODER,preprocessing=PREPROCESSING,
    native=NATIVE,max_memory_mb=MAX_PREPARATION_MEMORY_MB)
print('Run 분할:',{split:len(runs) for split,runs in PREPARED.split_runs.items()})
display(PREPARED.tables['support'])
display(PREPARED.tables['sensors'])  # TRAIN 기준 센서 제외 사유


In [ ]:
SENSOR_ROLES=training.sensor_roles(PREPARED)
if SENSOR_ROLES.empty or not SENSOR_ROLES.role.eq('prediction_target').any():
    raise ValueError('지원되는 판별 대상 센서가 없습니다. 위 전처리 결과를 확인하세요.')
TARGET_SENSOR_TABLE=SENSOR_ROLES.loc[SENSOR_ROLES.role.eq('prediction_target')].copy()
CONTEXT_SENSOR_TABLE=SENSOR_ROLES.loc[SENSOR_ROLES.role.eq('context_only')].copy()
display(SENSOR_ROLES.groupby(['step','role']).size().unstack(fill_value=0))
print('판별 대상:')
display(TARGET_SENSOR_TABLE)
print('문맥 입력 — 이 센서에는 질문·이상 주입을 적용하지 않습니다:')
display(CONTEXT_SENSOR_TABLE)


## 헤드 학습과 Test 평가

다음 셀은 실제 학습을 수행합니다. 세 방식이 같은 TRAIN/Validation/Test와 고정 GEVIS 특징을 사용합니다.
Validation으로 epoch를 선택하고, 세 헤드의 학습이 끝난 후 Test를 평가합니다.

- `softmax`: 기본 5유형 단일 분류
- `grouped`: 기본 설정에서는 Mean level과 Trend slope를 묶어서 학습
- `sigmoid`: 유형별 독립 BCE; 0.5 이상인 유형이 없거나 여러 개일 수 있음

`choice_exact_macro_f1`은 5유형 점수입니다(grouped에서는 정의하지 않음).
`choice_common_group_macro_f1`은 세 방식을 같은 그룹으로 환산한 비교 점수입니다.
통제 주입 실험의 변화 정답으로 평가하며, 실제 장비 고장 정답을 새로 만드는 것은 아닙니다.

장비 Job ID가 다른 날짜에 재사용되어도, 실제 시작·종료 시각으로 구분한 Run 단위로 독립성을 검사합니다.
이전 전처리가 완료되어 있다면 아래 학습 셀만 다시 실행해 식별 정보를 수정할 수 있습니다.


In [ ]:
# 이전 셀의 PREPARED·GEVIS를 유지한 채 수정된 라이브러리만 갱신합니다.
# 정규화 배열은 재계산하지 않고, 재사용된 장비 Job ID를 실제 Run 식별자로 바로잡습니다.
import importlib
for module_name in ('pecvd.experiments.ald_preprocessing','pecvd.experiments.ald_head_experiments'):
    importlib.reload(importlib.import_module(module_name))
training=importlib.reload(training)
PREPARED=training.canonicalize_run_lineages(PREPARED,progress=print)

try:
    RESULTS=training.train_ald(ROOT,GEVIS,PREPARED,encoder=ENCODER,
        preprocessing=PREPROCESSING,native=NATIVE,training=TRAINING,
        choices=CHOICES,steps=TARGET_PHASES)
except training.PreparationSupportError as error:
    SUPPORT_AUDIT=error.audit
    display(error.summary)  # 사용할 수 없는 경우 정확한 제외 사유·개수
    raise
display(RESULTS.support_summary)
display(RESULTS.summary)
print('저장본:',RESULTS.checkpoints)


In [ ]:
SENSOR_RESULTS=pd.concat([report['details'].assign(mode=mode)
    for mode,report in RESULTS.reports.items()],ignore_index=True)
SENSOR_SUMMARY=pd.concat([report['sensor_summary'].assign(mode=mode)
    for mode,report in RESULTS.reports.items()],ignore_index=True)
display(SENSOR_SUMMARY)
print('평가 구간·센서별 결과 행:',len(SENSOR_RESULTS))
print('판별 결과에는 문맥 전용 센서를 포함하지 않습니다.')


## 센서별 답변 확인

아래 설정으로 방식·센서·공정 경로를 골라 볼 수 있습니다. `None`은 전체입니다.
각 행은 **평가에 사용된 한 구간의 한 센서**입니다. BOOLEAN이 No이면 CHOICE·ORDINAL은 비워 둡니다.
sigmoid는 감지 집합을 `choice_answer`에 표시하고, 비교용 top-1은 `choice_top1`에 별도로 둡니다.

밸브 상태를 참고 입력으로 넣는 것만으로 밸브와 압력·유량의 물리 관계를 학습했다고 입증되지는 않습니다.


In [ ]:
VIEW_MODE='softmax'       # 'softmax' / 'grouped' / 'sigmoid' / None
VIEW_SENSOR=None          # 센서 이름 또는 None
VIEW_PHASE=None           # 공정 경로의 step 값 또는 None
ONLY_CHANGES=False
MAX_DISPLAY_ROWS=100

VIEW=SENSOR_RESULTS.copy()
if VIEW_MODE is not None:VIEW=VIEW.loc[VIEW['mode'].eq(VIEW_MODE)]
if VIEW_SENSOR is not None:VIEW=VIEW.loc[VIEW.center_sensor.eq(VIEW_SENSOR)]
if VIEW_PHASE is not None:VIEW=VIEW.loc[VIEW.step.eq(VIEW_PHASE)]
if ONLY_CHANGES:VIEW=VIEW.loc[VIEW.boolean_prediction.eq(1)]
print('선택된 결과:',len(VIEW),'행 | 아래 표는 최대',MAX_DISPLAY_ROWS,'행')
display(VIEW[['mode','run_id','step','offset_samples','center_sensor',
    'boolean_answer','boolean_yes_score','choice_answer','choice_top1','ordinal_answer',
    'injected','intervention_family']].head(MAX_DISPLAY_ROWS))
# 전체 결과는 SENSOR_RESULTS, 현재 필터의 전체 결과는 VIEW에 있습니다.


In [ ]:
TYPE_NAMES={1:'Mean level',2:'Variability',3:'Trend slope',4:'Local change',5:'Delay'}
GROUP_NAMES={0:'BOOLEAN No',**{i:' / '.join(TYPE_NAMES[j] for j in group)
    for i,group in enumerate(CHOICE_GROUPS,1)}}
for mode,report in RESULTS.reports.items():
    print(mode,'| 공통 그룹 혼동행렬: 행=정답, 열=예측')
    display(report['common_group_confusion'].rename(index=GROUP_NAMES,columns=GROUP_NAMES))
# 지원하지 못한 구간 및 사유: RESULTS.support_audit


헤드는 `artifacts/gevis_heads/peald`에 저장합니다.
재사용: `runtime=training.load_head(ROOT, checkpoint, gevis=GEVIS)`.
실시간 추론: `training.Session(runtime).process_window(...)`.
`standardized=False`이면 raw window에도 같은 밸브 인코딩·정규화를 적용하며,
기본 판별 대상은 저장된 연속형 센서 목록입니다. 문맥 전용 센서는 입력에 포함하지만 판별 결과를 만들지 않습니다.


In [ ]:
import importlib
import pecvd.experiments.ald_head_experiments as head_module
importlib.reload(head_module)
training = importlib.reload(training)

import time
import numpy as np
import pandas as pd
import torch

N_WINDOWS = 20

def sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)

# 공정별 Test Window를 번갈아 선택
pools = list(RESULTS.prepared.windows.items())
candidates = []
for i in range(max(len(p["test"]) for _, p in pools)):
    for step, pool in pools:
        if i < len(pool["test"]):
            candidates.append((step, pool["test"][i]))

rows = []

for mode, checkpoint in RESULTS.checkpoints.items():
    runtime = training.load_head(ROOT, checkpoint, gevis=GEVIS, allow_legacy_ald=True)
    session = training.Session(runtime)
    device = runtime.module.DEVICE

    def predict(step, w):
        return session.process_window(
            current=w["current"],
            step=step,
            recipe_key=w["cohort"][0],
            loop_key=str(w["cohort"][1]),
            offset_samples=int(w["cohort"][-1]),
            lineage_id=w["lineage_id"],
            run_id=w["run_id"],
            sensor_columns=w["sensor_columns"],
            standardized=True,
        )

    # 지원되는 동일 Window만 측정하고, 첫 호출은 워밍업으로 제외
    selected = []
    for step, w in candidates:
        reply = predict(step, w)
        if reply["status"] == "OK":
            selected.append((step, w))
        if len(selected) >= N_WINDOWS:
            break

    if not selected:
        raise RuntimeError(f"{mode}: 추론 가능한 Test Window가 없습니다.")

    # 각 입력 shape를 한 번 워밍업한 뒤 실제 시간 측정
    times = []
    for step, w in selected:
        sync(device)
        start = time.perf_counter()
        reply = predict(step, w)
        sync(device)
        elapsed = time.perf_counter() - start

        if reply["status"] != "OK":
            raise RuntimeError(reply)

        times.append(elapsed)

    rows.append({
        "방식": mode,
        "측정 Window 수": len(times),
        "총 추론 시간(s)": sum(times),
        "평균(s/Window)": np.mean(times),
        "중앙값(s/Window)": np.median(times),
        "P95(s/Window)": np.percentile(times, 95),
        "장치": str(device),
    })

INFERENCE_TIMING = pd.DataFrame(rows)
display(INFERENCE_TIMING.round(6))